# MVP 2 — Segmentação de Equipamentos por Perfil de Falha

**Nome:** _Seu nome aqui_  
**Matrícula:** _Sua matrícula aqui_  
**Data:** _dd/mm/aaaa_  
**Dataset:** Ordens de Serviço de Garantia — Imbera Brasil (2023–2026)  
**Tipo de problema:** Clusterização (Aprendizado Não Supervisionado)  

---

## Checklist do MVP

| Item | Status |
|---|---|
| Problema definido com contexto, objetivo e tipo de tarefa | ☐ |
| Dataset descrito, com fonte, atributos e restrições | ☐ |
| Dataset carregado por URL pública ou fonte diretamente acessível | ☐ |
| Análise exploratória objetiva, conectada à modelagem | ☐ |
| Features de agrupamento criadas e justificadas | ☐ |
| Normalização aplicada antes da clusterização | ☐ |
| Método do cotovelo para escolha do K (K-Means) | ☐ |
| Silhouette Score calculado | ☐ |
| K-Means aplicado e clusters interpretados | ☐ |
| DBSCAN aplicado para detecção de anomalias | ☐ |
| Perfil de cada cluster descrito com linguagem de negócio | ☐ |
| Recomendações para engenharia de produto | ☐ |
| Código limpo, organizado e executável do início ao fim | ☐ |
| Conclusão conectada ao objetivo inicial | ☐ |

---
# 1. Definição do Problema

## 1.1 Descrição do problema

A Imbera Brasil fabrica congeladores e freezers comerciais distribuídos para clientes dos segmentos Foodservice, KOF e terceiros, com garantia de 3 anos. Ao longo desse período, os equipamentos geram Ordens de Serviço (OS) de manutenção corretiva, com custos de peças, mão de obra e despesas adicionais.

Atualmente, cada OS é tratada de forma individual — não existe uma visão consolidada de **padrões de falha por perfil de equipamento**. Isso limita a capacidade da engenharia de produto de identificar problemas recorrentes e atuar preventivamente no projeto dos equipamentos.

**Usuários da solução:** engenharia de produto, qualidade e pós-venda da Imbera Brasil.  
**Relevância:** identificar grupos naturais de falha permite priorizar melhorias de projeto, antecipar manutenções e reduzir o custo total de garantia.

## 1.2 Objetivo do MVP

> O objetivo deste MVP é aplicar técnicas de clusterização (K-Means e DBSCAN) para **segmentar equipamentos da Imbera Brasil por perfil de falha**, descobrindo grupos naturais com base em características como modelo do produto, idade, tipo de defeito e custo acumulado de garantia — e traduzir esses grupos em recomendações acionáveis para a engenharia de produto.

**Objetivo deste trabalho:**  
> _Preencha aqui com suas palavras._

## 1.3 Tipo de problema

**Tipo escolhido:** Clusterização (Aprendizado Não Supervisionado)  
**Justificativa:** Não existe um target predefinido — não sabemos de antemão quantos grupos existem nem quais são. O objetivo é descobrir estruturas naturais nos dados. Por isso, a clusterização é a abordagem correta: ela agrupa equipamentos similares sem necessidade de rótulos prévios.

## 1.4 Premissas, hipóteses e critérios de sucesso

**Hipóteses iniciais:**
1. Equipamentos no 1º ano de garantia tendem a apresentar defeitos de fabricação e instalação (controlador, porta, ajustes).
2. Equipamentos no 2º e 3º ano tendem a apresentar defeitos de desgaste (motor, termostato, filtro secador, compressor).
3. Modelos diferentes (EVZ21, EVF19) têm perfis de falha distintos.
4. Existe um grupo de equipamentos com comportamento anômalo — custo muito alto ou frequência de OS muito acima da média.

**Critérios de sucesso:**
- Silhouette Score > 0.3 (clusters minimamente separados)
- Clusters interpretáveis com linguagem de negócio
- Pelo menos uma recomendação acionável por cluster
- DBSCAN identifica equipamentos anômalos para investigação

---
# 2. Ambiente, Bibliotecas e Reprodutibilidade

In [ ]:
import sys
import warnings
import random
import io
import requests
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print('Python:', sys.version.split()[0])
print('Seed:', SEED)

---
# 3. Carga dos Dados

## 3.1 Fonte dos dados

- **Dataset:** Ordens de Serviço de Garantia — Imbera Brasil
- **Período:** Janeiro/2023 a Abril/2026
- **Volume:** ~90.000 registros
- **Fonte:** Sistema interno de gestão de pós-venda da Imbera
- **Restrições:** Dados confidenciais — não publicar em repositórios abertos.
- **Nota:** Este é o mesmo dataset utilizado no MVP 1 (Previsão de Custo de Garantia), agora explorado sob a perspectiva de clusterização.

In [ ]:
# === Carga dos dados direto do GitHub ===
DATE_COL = 'Mês Fechamento - KOF'
COST_COL = 'SOMA DE GASTOS'

url = 'https://github.com/marcossilvalopesimbera-rgb/mvp-custo-garantia/raw/main/base_garantia_MVP.xlsx'

print('Carregando base do GitHub...')
response = requests.get(url, allow_redirects=True)

def limpar_moeda(val):
    if pd.isna(val): return np.nan
    if isinstance(val, (int, float)): return float(val)
    s = str(val).strip()
    s = re.sub(r'R\$', '', s)
    s = re.sub(r'\s+', '', s)
    s = s.replace('.', '').replace(',', '.').replace('-', '0')
    try: return float(s)
    except: return np.nan

df_raw = pd.read_excel(io.BytesIO(response.content), sheet_name='base_os', engine='openpyxl')

for col in ['Gastos com peças', 'MÃO DE OBRA', 'Valor Adicional', COST_COL]:
    df_raw[col] = df_raw[col].apply(limpar_moeda)

df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL], errors='coerce')
df_raw = df_raw.dropna(subset=[DATE_COL])
df_raw = df_raw[df_raw[COST_COL] >= 0]

print(f'✅ Base carregada com {len(df_raw)} linhas')
print(f'Período: {df_raw[DATE_COL].min().strftime("%b/%Y")} a {df_raw[DATE_COL].max().strftime("%b/%Y")}')
df_raw.head(3)

---
# 4. Engenharia de Features por Equipamento

Para clusterizar equipamentos, precisamos agregar as OS por número de série — transformando múltiplas OS em um único perfil por equipamento.

In [ ]:
# === Calcular idade do equipamento ===
df = df_raw.copy()

# Reconstruir data de fabricação a partir de Ano e Mês
df['data_fabricacao'] = pd.to_datetime(
    df['Ano'].astype(str) + '-' +
    df['Mês'].astype(str).str.zfill(2) + '-01',
    errors='coerce'
)

# Idade em meses na data do atendimento
df['idade_meses'] = (
    (df[DATE_COL].dt.year  - df['data_fabricacao'].dt.year) * 12 +
    (df[DATE_COL].dt.month - df['data_fabricacao'].dt.month)
).clip(0, 36)

# Ano de garantia (1, 2 ou 3)
df['ano_garantia'] = pd.cut(
    df['idade_meses'],
    bins=[0, 12, 24, 36],
    labels=[1, 2, 3],
    include_lowest=True
).astype(float)

print('Distribuição por ano de garantia:')
print(df['ano_garantia'].value_counts().sort_index())

In [ ]:
# === Agregar por equipamento (n° de série) ===

# Defeito mais frequente por equipamento
defeito_principal = (
    df.groupby('n° de série')['Defeito Constatado']
    .agg(lambda x: x.value_counts().index[0] if len(x) > 0 else 'Desconhecido')
    .reset_index()
    .rename(columns={'Defeito Constatado': 'defeito_principal'})
)

# Agregação principal
df_equip = (
    df.groupby('n° de série')
    .agg(
        modelo        = ('Produto ajustado', lambda x: x.value_counts().index[0]),
        familia       = ('FAMÍLIA DO PRODUTO', lambda x: x.value_counts().index[0]),
        cliente_seg   = ('Classificação cliente', lambda x: x.value_counts().index[0]),
        total_os      = (COST_COL, 'count'),
        custo_total   = (COST_COL, 'sum'),
        custo_medio   = (COST_COL, 'mean'),
        custo_max     = (COST_COL, 'max'),
        idade_media   = ('idade_meses', 'mean'),
        ano_garantia  = ('ano_garantia', lambda x: x.mode()[0] if len(x) > 0 else np.nan),
        tipos_defeito = ('Defeito Constatado', 'nunique'),
    )
    .reset_index()
)

# Juntar defeito principal
df_equip = df_equip.merge(defeito_principal, on='n° de série', how='left')

# Remover equipamentos sem dados válidos
df_equip = df_equip.dropna(subset=['custo_total', 'idade_media']).reset_index(drop=True)

print(f'Equipamentos únicos: {len(df_equip)}')
print(f'\nEstatísticas:')
display(df_equip[['total_os', 'custo_total', 'custo_medio', 'idade_media']].describe().round(2))

---
# 5. Análise Exploratória

In [ ]:
# === Gráfico 1: Distribuição de OS por modelo ===
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_equip['modelo'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Equipamentos por Modelo')
axes[0].set_xlabel('Modelo')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=45)

df_equip['ano_garantia'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('Equipamentos por Ano de Garantia')
axes[1].set_xlabel('Ano de Garantia')
axes[1].set_ylabel('Quantidade')

axes[2].hist(df_equip['custo_total'], bins=40, color='tomato', edgecolor='white')
axes[2].set_title('Distribuição do Custo Total por Equipamento')
axes[2].set_xlabel('Custo Total (R$)')
axes[2].set_ylabel('Frequência')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}k'))

plt.tight_layout()
plt.show()

In [ ]:
# === Gráfico 2: Custo médio por ano de garantia e modelo ===
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

custo_garantia = df_equip.groupby('ano_garantia')['custo_medio'].mean()
custo_garantia.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Custo Médio por OS — por Ano de Garantia')
axes[0].set_xlabel('Ano de Garantia')
axes[0].set_ylabel('Custo Médio (R$)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))

custo_modelo = df_equip.groupby('modelo')['custo_total'].mean().sort_values(ascending=False)
custo_modelo.plot(kind='bar', ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('Custo Total Médio por Modelo')
axes[1].set_xlabel('Modelo')
axes[1].set_ylabel('Custo Total Médio (R$)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# === Gráfico 3: Top 10 defeitos mais frequentes ===
top_defeitos = df['Defeito Constatado'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 4))
top_defeitos.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 10 Defeitos Mais Frequentes')
ax.set_xlabel('Quantidade de OS')
plt.tight_layout()
plt.show()

## 5.1 Síntese da análise exploratória

**Preencha após observar os gráficos acima:**

- Qual modelo concentra mais equipamentos e maior custo?
- O custo médio aumenta com o ano de garantia? Isso confirma a hipótese de desgaste progressivo?
- Quais são os defeitos mais frequentes? Estão concentrados em algum modelo ou ano de garantia?
- Há indícios visuais de grupos naturais nos dados?

**Síntese:**  
> _Preencha aqui._

---
# 6. Preparação para Clusterização

In [ ]:
# === Selecionar e encodar features ===

# Encodar variáveis categóricas
le_modelo  = LabelEncoder()
le_defeito = LabelEncoder()
le_cliente = LabelEncoder()

df_feat = df_equip.copy()
df_feat['modelo_enc']   = le_modelo.fit_transform(df_feat['modelo'].fillna('Desconhecido'))
df_feat['defeito_enc']  = le_defeito.fit_transform(df_feat['defeito_principal'].fillna('Desconhecido'))
df_feat['cliente_enc']  = le_cliente.fit_transform(df_feat['cliente_seg'].fillna('Desconhecido'))

# Features para clusterização
FEATURE_COLS = [
    'total_os',       # frequência de manutenção
    'custo_total',    # custo acumulado de garantia
    'custo_medio',    # ticket médio por OS
    'custo_max',      # pior caso
    'idade_media',    # idade média do equipamento
    'ano_garantia',   # ano de garantia (1, 2 ou 3)
    'tipos_defeito',  # diversidade de defeitos
    'modelo_enc',     # modelo do produto
    'defeito_enc',    # defeito principal
    'cliente_enc',    # segmento do cliente
]

X = df_feat[FEATURE_COLS].fillna(0)

# Normalizar — OBRIGATÓRIO para K-Means e DBSCAN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Shape para clusterização: {X_scaled.shape}')
print(f'Features: {FEATURE_COLS}')

## 6.1 Justificativa das features

| Feature | Justificativa |
|---|---|
| total_os | Frequência de manutenção — equipamentos problemáticos têm mais OS |
| custo_total | Impacto financeiro acumulado do equipamento |
| custo_medio | Ticket médio — reflete gravidade dos defeitos |
| custo_max | Pior evento — identifica equipamentos com falhas graves |
| idade_media | Idade na data do atendimento — captura padrão de desgaste |
| ano_garantia | Ano de garantia (1, 2, 3) — período da vida útil |
| tipos_defeito | Diversidade de defeitos — equipamentos com múltiplos problemas |
| modelo_enc | Modelo do produto — perfis diferentes por linha |
| defeito_enc | Defeito mais frequente — tipo de falha predominante |
| cliente_enc | Segmento do cliente — uso diferente por canal |

> **Por que normalizar?** K-Means e DBSCAN usam distância euclidiana. Sem normalização, `custo_total` (valores em R$ milhares) dominaria completamente sobre `total_os` (valores unitários), tornando as outras features irrelevantes.

---
# 7. K-Means — Método do Cotovelo

In [ ]:
# === Método do Cotovelo ===
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(K_range, inertias, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Método do Cotovelo — Inertia vs K')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].grid(alpha=0.3)

axes[1].plot(K_range, silhouettes, marker='s', color='seagreen', linewidth=2)
axes[1].set_title('Silhouette Score vs K')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].axhline(y=0.3, linestyle='--', color='tomato', alpha=0.7, label='Mínimo aceitável (0.3)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\nSilhouette Score por K:')
for k, s in zip(K_range, silhouettes):
    print(f'  K={k}: {s:.4f}')

## 7.1 Escolha do K

**Preencha após observar os gráficos:**

- Onde está o cotovelo na curva de inertia?
- Qual K tem o maior Silhouette Score?
- O K escolhido faz sentido para o negócio?

**K escolhido:** _preencha_  
**Justificativa:** _preencha_

---
# 8. K-Means — Treinamento e Interpretação

In [ ]:
# === Treinar K-Means com o K escolhido ===
# TODO: ajuste K_FINAL com base no método do cotovelo
K_FINAL = 4

kmeans = KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=10)
df_feat['cluster_km'] = kmeans.fit_predict(X_scaled)

score = silhouette_score(X_scaled, df_feat['cluster_km'])
print(f'K-Means com K={K_FINAL}')
print(f'Silhouette Score: {score:.4f}')
print(f'\nDistribuição dos clusters:')
print(df_feat['cluster_km'].value_counts().sort_index())

In [ ]:
# === Perfil de cada cluster ===
perfil = df_feat.groupby('cluster_km').agg(
    n_equipamentos   = ('n° de série', 'count'),
    os_media         = ('total_os', 'mean'),
    custo_total_medio= ('custo_total', 'mean'),
    custo_os_medio   = ('custo_medio', 'mean'),
    idade_media      = ('idade_media', 'mean'),
    ano_garantia     = ('ano_garantia', 'mean'),
    tipos_defeito    = ('tipos_defeito', 'mean'),
).round(2)

print('=== Perfil dos Clusters ===')
display(perfil)

print('\n=== Modelo mais frequente por cluster ===')
display(df_feat.groupby('cluster_km')['modelo'].agg(lambda x: x.value_counts().index[0]).to_frame('modelo_predominante'))

print('\n=== Defeito principal por cluster ===')
display(df_feat.groupby('cluster_km')['defeito_principal'].agg(lambda x: x.value_counts().index[0]).to_frame('defeito_predominante'))

## 8.1 Interpretação dos clusters

**Preencha com base nas tabelas acima — dê um nome e uma descrição para cada cluster:**

| Cluster | Nome sugerido | Perfil | Recomendação para engenharia |
|---|---|---|---|
| 0 | _preencha_ | _preencha_ | _preencha_ |
| 1 | _preencha_ | _preencha_ | _preencha_ |
| 2 | _preencha_ | _preencha_ | _preencha_ |
| 3 | _preencha_ | _preencha_ | _preencha_ |

> **Dica:** use a tabela de perfil para identificar o que diferencia cada cluster. Ex: Cluster 0 pode ser "equipamentos novos com baixo custo" e Cluster 3 pode ser "equipamentos velhos com alto custo e múltiplos defeitos".

---
# 9. Visualização dos Clusters

In [ ]:
# === PCA para visualização 2D ===
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

variancia = pca.explained_variance_ratio_
print(f'PC1 explica: {variancia[0]*100:.1f}%')
print(f'PC2 explica: {variancia[1]*100:.1f}%')
print(f'Total: {sum(variancia)*100:.1f}% da variância')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot K-Means
colors = ['steelblue', 'seagreen', 'tomato', 'orange', 'purple', 'brown']
for c in range(K_FINAL):
    mask = df_feat['cluster_km'] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors[c], label=f'Cluster {c}',
                   alpha=0.5, s=20)
axes[0].set_title(f'K-Means (K={K_FINAL}) — Visualização PCA')
axes[0].set_xlabel(f'PC1 ({variancia[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({variancia[1]*100:.1f}%)')
axes[0].legend()

# Plot por ano de garantia
cores_garantia = {1.0: 'seagreen', 2.0: 'steelblue', 3.0: 'tomato'}
for ano, cor in cores_garantia.items():
    mask = df_feat['ano_garantia'] == ano
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=cor, label=f'Ano {int(ano)}',
                   alpha=0.5, s=20)
axes[1].set_title('Distribuição por Ano de Garantia — PCA')
axes[1].set_xlabel(f'PC1 ({variancia[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({variancia[1]*100:.1f}%)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# === Radar chart — perfil dos clusters ===
from matplotlib.patches import FancyArrowPatch

features_radar = ['os_media', 'custo_total_medio', 'custo_os_medio', 'idade_media', 'tipos_defeito']
labels_radar   = ['Freq. OS', 'Custo Total', 'Custo/OS', 'Idade', 'Tipos Defeito']

# Normalizar para 0-1 para visualização
perfil_norm = perfil[features_radar].copy()
for col in features_radar:
    mn, mx = perfil_norm[col].min(), perfil_norm[col].max()
    perfil_norm[col] = (perfil_norm[col] - mn) / (mx - mn + 1e-9)

N = len(features_radar)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, K_FINAL, figsize=(4*K_FINAL, 4), subplot_kw=dict(polar=True))
if K_FINAL == 1: axes = [axes]

for i, (idx, row) in enumerate(perfil_norm.iterrows()):
    vals = row.tolist() + row.tolist()[:1]
    axes[i].plot(angles, vals, color=colors[i], linewidth=2)
    axes[i].fill(angles, vals, color=colors[i], alpha=0.25)
    axes[i].set_xticks(angles[:-1])
    axes[i].set_xticklabels(labels_radar, size=9)
    axes[i].set_title(f'Cluster {idx}', size=12, pad=15)
    axes[i].set_ylim(0, 1)

plt.suptitle('Perfil dos Clusters — Radar Chart', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

---
# 10. DBSCAN — Detecção de Anomalias

In [ ]:
# === K-Distance Graph para escolha do eps ===
k_dist = 5
nbrs = NearestNeighbors(n_neighbors=k_dist).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
distances = np.sort(distances[:, k_dist-1])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(distances, color='steelblue', linewidth=1.5)
ax.set_title(f'K-Distance Graph (k={k_dist}) — Escolha do eps')
ax.set_xlabel('Pontos ordenados')
ax.set_ylabel(f'Distância ao {k_dist}º vizinho')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Observe o "joelho" do gráfico — esse valor sugere o eps ideal.')
print(f'Sugestão automática (percentil 95%): {np.percentile(distances, 95):.3f}')

In [ ]:
# === Aplicar DBSCAN ===
# TODO: ajuste EPS com base no K-Distance Graph acima
EPS = np.percentile(distances, 95)
MIN_SAMPLES = 5

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
df_feat['cluster_db'] = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(df_feat['cluster_db'])) - (1 if -1 in df_feat['cluster_db'].values else 0)
n_outliers    = (df_feat['cluster_db'] == -1).sum()

print(f'DBSCAN — eps={EPS:.3f}, min_samples={MIN_SAMPLES}')
print(f'Clusters encontrados: {n_clusters_db}')
print(f'Outliers (anomalias): {n_outliers} ({n_outliers/len(df_feat)*100:.1f}%)')

In [ ]:
# === Análise dos equipamentos anômalos ===
anomalias = df_feat[df_feat['cluster_db'] == -1].copy()

print(f'=== {len(anomalias)} Equipamentos com Comportamento Anômalo ===')
print(f'\nEstatísticas dos anômalos vs população geral:')

comparacao = pd.DataFrame({
    'Anômalos': anomalias[['total_os','custo_total','custo_medio','idade_media']].mean(),
    'Geral':    df_feat[['total_os','custo_total','custo_medio','idade_media']].mean()
}).round(2)
display(comparacao)

print('\nModelos mais frequentes entre anômalos:')
print(anomalias['modelo'].value_counts().head(5))

print('\nDefeitos mais frequentes entre anômalos:')
print(anomalias['defeito_principal'].value_counts().head(5))

print('\nTop 10 equipamentos com maior custo total:')
display(anomalias.nlargest(10, 'custo_total')[['n° de série','modelo','custo_total','total_os','defeito_principal','ano_garantia']])

In [ ]:
# === Visualização DBSCAN ===
fig, ax = plt.subplots(figsize=(10, 6))

mask_normal  = df_feat['cluster_db'] != -1
mask_anomaly = df_feat['cluster_db'] == -1

ax.scatter(X_pca[mask_normal,  0], X_pca[mask_normal,  1],
           c='steelblue', alpha=0.4, s=15, label='Normal')
ax.scatter(X_pca[mask_anomaly, 0], X_pca[mask_anomaly, 1],
           c='tomato', alpha=0.8, s=40, marker='X', label=f'Anômalo ({n_outliers})')

ax.set_title('DBSCAN — Detecção de Equipamentos Anômalos')
ax.set_xlabel(f'PC1 ({variancia[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({variancia[1]*100:.1f}%)')
ax.legend()
plt.tight_layout()
plt.show()

## 10.1 Interpretação das anomalias

**Preencha após analisar os resultados do DBSCAN:**

- Quantos equipamentos foram identificados como anômalos?
- Qual o perfil típico desses equipamentos (modelo, idade, defeito)?
- O custo médio dos anômalos é significativamente maior que o geral?
- Que ação a engenharia de produto deveria tomar com esses equipamentos?

**Resposta:**  
> _Preencha aqui._

---
# 11. Comparação Final dos Modelos

| Modelo | Clusters | Silhouette | Outliers | Observações |
|---|---|---|---|---|
| K-Means | _preencha_ | _preencha_ | N/A | Grupos bem definidos |
| DBSCAN | _preencha_ | N/A | _preencha_ | Detecção de anomalias |

> **Complementaridade:** K-Means e DBSCAN se complementam — K-Means encontra os grupos principais, DBSCAN identifica os outliers que não se encaixam em nenhum grupo.

---
# 12. Recomendações para Engenharia de Produto

**Preencha com base nos clusters identificados:**

| Cluster | Perfil | Ação recomendada |
|---|---|---|
| _preencha_ | _preencha_ | _preencha_ |
| _preencha_ | _preencha_ | _preencha_ |
| Anômalos (DBSCAN) | Alto custo, múltiplos defeitos | Inspeção individual, possível recall de lote |

**Próximos passos sugeridos:**
1. Cruzar os clusters com dados de lote de fabricação — identificar se problemas são sistêmicos
2. Usar os clusters como base para o **MVP 3 — Previsão de Reincidência**
3. Implementar alerta automático para equipamentos com perfil do cluster de alto risco
4. Revisar projeto das peças mais frequentes nos clusters problemáticos

---
# 13. Conclusão

**Preencha após executar o notebook com os resultados reais:**

- **Objetivo:** segmentar equipamentos da Imbera por perfil de falha para retroalimentar a engenharia de produto.
- **Melhor solução:** K-Means com K=___ (Silhouette Score = ___) + DBSCAN para anomalias.
- **Principais descobertas:** _descreva o que cada cluster representa._
- **Anomalias:** ___ equipamentos com comportamento fora do padrão identificados.
- **Limitações:** clusterização é sensível à escolha de features e normalização; resultados devem ser validados com especialistas de produto.
- **Próximos passos:** usar os clusters como features no MVP 3 — Previsão de Reincidência de Defeito.

**Conclusão:**  
> _Preencha aqui._